In [1]:
# Installs
'''
pip install -q "qdrant-client[fastembed]>=1.14.2"

docker pull qdrant/qdrant

docker run -p 6333:6333 -p 6334:6334 \
   -v "$(pwd)/qdrant_storage:/qdrant/storage:z" \
   qdrant/qdrant
'''

'\npip install -q "qdrant-client[fastembed]>=1.14.2"\n\ndocker pull qdrant/qdrant\n\ndocker run -p 6333:6333 -p 6334:6334    -v "$(pwd)/qdrant_storage:/qdrant/storage:z"    qdrant/qdrant\n'

In [2]:
from fastembed import TextEmbedding
from qdrant_client import QdrantClient, models

/home/codespace/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Q1

In [3]:
# Setup Quadrant client, select model
qd_client = QdrantClient("http://localhost:6333") #connecting to local Qdrant instance
EMBEDDING_DIMENSIONALITY = 512
model_handle = "jinaai/jina-embeddings-v2-small-en"
model = TextEmbedding(model_name=model_handle)

Fetching 5 files: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████| 5/5 [00:03<00:00,  1.40it/s]


In [4]:
query = 'I just discovered the course. Can I join now?'

In [5]:
query_vec = list(model.embed(query))
min(query_vec[0])

np.float64(-0.11726373885183883)

# Q2

In [6]:
import numpy as np
np.linalg.norm(query_vec)

np.float64(1.0)

In [7]:
doc = 'Can I still join the course after the start date?'
doc_vec = list(model.embed(doc))
query_vec[0].dot(doc_vec[0])

np.float64(0.9008528895674548)

# Q3

In [8]:
documents = [{'text': "Yes, even if you don't register, you're still eligible to submit the homeworks.\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything for the last minute.",
  'section': 'General course-related questions',
  'question': 'Course - Can I still join the course after the start date?',
  'course': 'data-engineering-zoomcamp'},
 {'text': 'Yes, we will keep all the materials after the course finishes, so you can follow the course at your own pace after it finishes.\nYou can also continue looking at the homeworks and continue preparing for the next cohort. I guess you can also start working on your final capstone project.',
  'section': 'General course-related questions',
  'question': 'Course - Can I follow the course after it finishes?',
  'course': 'data-engineering-zoomcamp'},
 {'text': "The purpose of this document is to capture frequently asked technical questions\nThe exact day and hour of the course will be 15th Jan 2024 at 17h00. The course will start with the first  “Office Hours'' live.1\nSubscribe to course public Google Calendar (it works from Desktop only).\nRegister before the course starts using this link.\nJoin the course Telegram channel with announcements.\nDon’t forget to register in DataTalks.Club's Slack and join the channel.",
  'section': 'General course-related questions',
  'question': 'Course - When will the course start?',
  'course': 'data-engineering-zoomcamp'},
 {'text': 'You can start by installing and setting up all the dependencies and requirements:\nGoogle cloud account\nGoogle Cloud SDK\nPython 3 (installed with Anaconda)\nTerraform\nGit\nLook over the prerequisites and syllabus to see if you are comfortable with these subjects.',
  'section': 'General course-related questions',
  'question': 'Course - What can I do before the course starts?',
  'course': 'data-engineering-zoomcamp'},
 {'text': 'Star the repo! Share it with friends if you find it useful ❣️\nCreate a PR if you see you can improve the text or the structure of the repository.',
  'section': 'General course-related questions',
  'question': 'How can we contribute to the course?',
  'course': 'data-engineering-zoomcamp'}]

In [9]:
docs_text = [doc['text'] for i, doc in enumerate(documents)]
doc_vecs = np.array(list(model.embed(docs_text)))
print(doc_vecs.shape)

(5, 512)


In [10]:
cos_sims = doc_vecs.dot(query_vec[0])
print(cos_sims)
print(np.argmax(cos_sims))

[0.76296847 0.81823782 0.80853974 0.7133079  0.73044992]
1


# Q4

In [11]:
docs_text = [doc['question'] + ' ' + doc['text'] for i, doc in enumerate(documents)]
doc_vecs = np.array(list(model.embed(docs_text)))
cos_sims = doc_vecs.dot(query_vec[0])
print(cos_sims)
print(np.argmax(cos_sims))

[0.85145432 0.84365942 0.8408287  0.7755158  0.80860078]
0


# Q5

In [12]:
import json

model_dim_min = 999999
for model in TextEmbedding.list_supported_models():
    if model["dim"] <= model_dim_min:
        model_dim_min = model["dim"]
        # print(json.dumps(model, indent=2))
EMBEDDING_DIMENSIONALITY = model_dim_min
print(EMBEDDING_DIMENSIONALITY)

384


In [13]:
for model in TextEmbedding.list_supported_models():
    if model["dim"] <= EMBEDDING_DIMENSIONALITY:
        print(json.dumps(model, indent=2))

{
  "model": "BAAI/bge-small-en",
  "sources": {
    "hf": "Qdrant/bge-small-en",
    "url": "https://storage.googleapis.com/qdrant-fastembed/BAAI-bge-small-en.tar.gz",
    "_deprecated_tar_struct": true
  },
  "model_file": "model_optimized.onnx",
  "description": "Text embeddings, Unimodal (text), English, 512 input tokens truncation, Prefixes for queries/documents: necessary, 2023 year.",
  "license": "mit",
  "size_in_GB": 0.13,
  "additional_files": [],
  "dim": 384,
  "tasks": {}
}
{
  "model": "BAAI/bge-small-en-v1.5",
  "sources": {
    "hf": "qdrant/bge-small-en-v1.5-onnx-q",
    "url": null,
    "_deprecated_tar_struct": false
  },
  "model_file": "model_optimized.onnx",
  "description": "Text embeddings, Unimodal (text), English, 512 input tokens truncation, Prefixes for queries/documents: not so necessary, 2023 year.",
  "license": "mit",
  "size_in_GB": 0.067,
  "additional_files": [],
  "dim": 384,
  "tasks": {}
}
{
  "model": "snowflake/snowflake-arctic-embed-xs",
  "sou

In [14]:
model_handle = "BAAI/bge-small-en"

# Q6

In [15]:
import requests 

docs_url = 'https://github.com/alexeygrigorev/llm-rag-workshop/raw/main/notebooks/documents.json'
docs_response = requests.get(docs_url)
documents_raw = docs_response.json()
documents = []
for course in documents_raw:
    course_name = course['course']
    if course_name != 'machine-learning-zoomcamp':
        continue

    for doc in course['documents']:
        doc['course'] = course_name
        documents.append(doc)

In [17]:
collection_name = "hw2_docs"
qd_client.delete_collection(collection_name=collection_name)
qd_client.create_collection(
    collection_name=collection_name,
    vectors_config=models.VectorParams(
        size=EMBEDDING_DIMENSIONALITY,
        distance=models.Distance.COSINE
    )
)

True

In [18]:
qd_client.create_payload_index(
    collection_name=collection_name,
    field_name="course",
    field_schema="keyword"
)

UpdateResult(operation_id=1, status=<UpdateStatus.COMPLETED: 'completed'>)

In [20]:
qd_client.upsert(
    collection_name=collection_name,
    points=points
)

Fetching 5 files: 100%|██████████████████████████████████████████████████████████████████████████████████████| 5/5 [00:00<00:00,  6.57it/s]


UpdateResult(operation_id=2, status=<UpdateStatus.COMPLETED: 'completed'>)

In [29]:
points = []
for i, doc in enumerate(documents):
    text = doc['question'] + ' ' + doc['text']
    vector = models.Document(text=text, model=model_handle)
    point = models.PointStruct(
        id=i,
        vector=vector,
        payload=doc
    )
    points.append(point)

In [46]:
def vector_search(question):
    print('vector_search is used')
    
    # course = 'data-engineering-zoomcamp'
    query_points = qd_client.query_points(
        collection_name=collection_name,
        query=models.Document(
            text=question,
            model=model_handle 
        ),
        # query_filter=models.Filter( 
        #     must=[
        #         models.FieldCondition(
        #             key="course",
        #             match=models.MatchValue(value=course)
        #         )
        #     ]
        # ),
        limit=5,
        with_payload=True
    )
    
    results = []
    scores = []
    for point in query_points.points:
        results.append(point.payload)
        scores.append(point.score)
    
    return results, scores

In [47]:
query_results, scores = vector_search(query)

vector_search is used


In [48]:
scores

[0.8703172, 0.86918855, 0.86833113, 0.8576106, 0.85715395]